# Dysarthric Speech Detection on TORGO - GPU run

Detecting dysarthria (slurred speech caused by cerebral palsy or ALS) from a
short recording, for early screening.

This notebook runs the whole pipeline on a Colab GPU:

1. Download TORGO (academic-use licence - cite Rudzicz et al., 2012)
2. Build an utterance manifest, recording the microphone explicitly
3. eGeMAPS + XGBoost, and frozen WavLM + a small classifier
4. **Leave-one-speaker-out** over all 15 speakers, one microphone for both groups
5. SHAP explanation, severity breakdown, and the microphone-confound audit

**Runtime -> Change runtime type -> GPU** before running.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU"

In [ ]:
!pip -q install opensmile==2.6.0 xgboost shap soundfile librosa pyarrow transformers --upgrade

## Unpack the pipeline
The repository's `src/` files are embedded below.

In [ ]:
import base64, json, zlib, pathlib
_P = "..."
_files = json.loads(zlib.decompress(base64.b64decode(_P)).decode())
pathlib.Path("src").mkdir(exist_ok=True); pathlib.Path("scripts").mkdir(exist_ok=True)
for _n, _t in _files.items():
    _d = "scripts" if _n.endswith(".sh") else "src"
    pathlib.Path(_d, _n).write_text(_t, encoding="utf-8")
print("wrote:", sorted(_files))

## 1. Download TORGO

Licence: free for **academic (non-profit)** use, and any publication using it
must cite Rudzicz et al. Only the audio and prompts are extracted; the
articulatory (.pos) and video data are discarded.

In [ ]:
!mkdir -p data/TORGO
!for S in F FC M MC; do bash scripts/fetch_torgo.sh data/TORGO $S; done
!ls data/TORGO

## 2. Manifest (microphone recorded explicitly)

In [ ]:
!python src/torgo_manifest.py --root data/TORGO --out data/manifest.csv

## 3. Features

eGeMAPS = 88 interpretable acoustic functionals (pitch, loudness, jitter,
shimmer, rate, spectral shape). WavLM is **frozen** - only a small classifier is
trained on top, so 15 speakers cannot be memorised by a 94M-parameter encoder.

In [ ]:
!python src/extract_egemaps.py --mic both --out features/egemaps.parquet

In [ ]:
!python src/extract_wavlm.py --mic both --out features/wavlm.parquet --batch-size 16

## 4. Leave-one-speaker-out
Primary protocol: head-mounted microphone for **both** groups.

In [ ]:
!python src/run_experiment.py --features features/egemaps.parquet --model xgb    --mic head --name egemaps_xgb
!python src/run_experiment.py --features features/wavlm.parquet   --model logreg --mic head --name wavlm_logreg
!python src/run_experiment.py --features features/wavlm.parquet   --model mlp    --mic head --name wavlm_mlp

## 5. Controls: array microphone, shared prompts, and no absolute loudness

In [ ]:
!python src/run_experiment.py --features features/egemaps.parquet --model xgb    --mic array --name egemaps_xgb_array
!python src/run_experiment.py --features features/wavlm.parquet   --model logreg --mic array --name wavlm_logreg_array
!python src/run_experiment.py --features features/egemaps.parquet --model xgb    --mic head --shared-prompts --name egemaps_xgb_sharedprompt
!python src/run_experiment.py --features features/wavlm.parquet   --model logreg --mic head --shared-prompts --name wavlm_logreg_sharedprompt
!python src/run_experiment.py --features features/egemaps.parquet --model xgb    --mic head --no-level-features --name egemaps_xgb_nolevel

## 6. Explanation, severity and the microphone audit

In [ ]:
!python src/explain_shap.py --features features/egemaps.parquet --mic head

In [ ]:
!python src/mic_confound.py --features features/egemaps.parquet

In [ ]:
!python src/analyze.py --outdir results
!python src/make_report.py --outdir results --out RESULTS.md

## 7. Download the results

In [ ]:
!zip -qr results.zip results RESULTS.md data/manifest.csv features/egemaps.parquet
from google.colab import files; files.download("results.zip")